In [1]:
pip install quantvn


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
from quantvn.vn.data.utils import client

client(apikey="Glt31lvLLZgzMWrbW1chLbNlkvhztKLWHIVlviEMlF4YJvYiY.V9GO1yYm9GUVI1Whti-_ZGWAquu0jR6flicEbiSw3GcDA0uccT12_165PfuNdZUzI5abgmOY-fWTQW")

/Users/user1/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
from quantvn.vn.data import get_stock_hist
import pandas as pd

df = get_stock_hist("VIC", resolution="1D")
print(df.tail(2))




           Date   Open   High    Low  Close   volume
4061 2023-12-28  21.80  22.30  21.80  22.22  4075667
4062 2023-12-29  22.32  22.42  22.22  22.30  2172536


In [4]:
import numpy as np
import pandas as pd

def gen_position(df: pd.DataFrame) -> pd.DataFrame:
    """
    Chiến lược MA Cross:
    - Mua (Position = 1) khi đường MA ngắn hạn cắt lên trên đường MA dài hạn.
    - Bán/Đứng ngoài (Position = 0) khi đường MA ngắn cắt xuống dưới MA dài.
    """
    # Tạo một bản sao để đảm bảo không làm hỏng dữ liệu gốc truyền vào
    df_strat = df.copy()
    
    # --- PHẦN 1: TÍNH TOÁN CHỈ BÁO ---
    # Thiết lập chu kỳ cho 2 đường Moving Average
    short_window = 20
    long_window = 50
    
    # Tính toán SMA (Simple Moving Average) dựa trên giá Đóng cửa
    df_strat['SMA_Short'] = df_strat['Close'].rolling(window=short_window).mean()
    df_strat['SMA_Long'] = df_strat['Close'].rolling(window=long_window).mean()
    
    # --- PHẦN 2: TẠO TÍN HIỆU VÀ VỊ THẾ GIAO DỊCH ---
    # Cột Signal: Đánh dấu 1 nếu MA Ngắn > MA Dài, ngược lại là 0
    df_strat['Signal'] = 0.0
    # Bắt đầu tính tín hiệu từ khi MA dài hạn có đủ dữ liệu
    df_strat.iloc[long_window:, df_strat.columns.get_loc('Signal')] = np.where(
        df_strat['SMA_Short'].iloc[long_window:] > df_strat['SMA_Long'].iloc[long_window:], 1.0, 0.0
    )
    
    # Cột Position: Dịch chuyển (shift) tín hiệu đi 1 phiên
    # LÝ DO: Giả sử hôm nay đóng cửa có tín hiệu cắt nhau, bạn chỉ có thể vào lệnh ở phiên ngày mai.
    # Nếu không dùng shift(1), chiến lược sẽ bị dính lỗi "Look-ahead bias" (Nhìn trước tương lai).
    df_strat['position'] = df_strat['Signal'].shift(1).fillna(0.0)
    
    return df_strat